# Thesis Stage 3 — Stream Simulation, Detection Latency & Geoparsing Comparison


### Research questions addressed

- **RQ2 (primary):** What is the detection latency of the system, and how does it vary with the alert threshold?
- **RQ3 (primary):** Which geoparsing approach performs best on disaster text?
- **RQ4:** Does combining temporal + spatial detection improve event detection vs temporal-only?

### Pipeline
Raw tweet → preprocess → DistilBERT classify → **[Geoparser A | Geoparser B]** → Z-score/CUSUM burst detect → DBSCAN spatial cluster → Combined alert.

In [ ]:
# Cell 1: Environment Setup + Drive Mount
!pip install transformers torch scikit-learn pandas numpy \
             matplotlib scipy wordsegment emoji gliner \
             --quiet

import os, json, math, time, warnings, re, unicodedata, shutil, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict, Counter
from datetime import datetime, timedelta

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.cluster import DBSCAN

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.dpi"        : 150,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "axes.titlesize"    : 12,
    "axes.labelsize"    : 10,
    "xtick.labelsize"   : 9,
    "ytick.labelsize"   : 9,
    "font.family"       : "DejaVu Sans",
})
DISASTER_COLOR   = "#D85A30"
NODISASTER_COLOR = "#1D9E75"
NEUTRAL_COLOR    = "#7F77DD"
ALERT_COLOR      = "#E8A838"
NOISE_COLOR      = "#888888"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU  : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("  ⚠  CPU mode — transformer inference will be slow")
    print("     Runtime → Change runtime type → T4 GPU  (strongly recommended)")

from google.colab import drive
drive.mount("/content/drive")

DRIVE_STAGE1_DIR = "/content/drive/MyDrive/Thesis_Stage1"
DRIVE_STAGE2_DIR = "/content/drive/MyDrive/Thesis_Stage2"
DRIVE_STAGE3_DIR = "/content/drive/MyDrive/Thesis_Stage3"
os.makedirs(DRIVE_STAGE3_DIR, exist_ok=True)

def save_to_drive(local_path, label=""):
    dest = os.path.join(DRIVE_STAGE3_DIR, os.path.basename(local_path))
    shutil.copy2(local_path, dest)
    print(f"  ✓ {os.path.basename(local_path)} → Drive  {label}")

SEED = 42
np.random.seed(SEED)

print(f"\n  Stage 1 artifacts : {DRIVE_STAGE1_DIR}")
print(f"  Stage 2 artifacts : {DRIVE_STAGE2_DIR}")
print(f"  Stage 3 outputs   : {DRIVE_STAGE3_DIR}")
print(f"  Seed              : {SEED}")
print("\n✓ Cell 1 complete")


In [ ]:
# Cell 2: Load Stage 2 Classifier + Preprocessing
# Loads the fine-tuned DistilBERT (threshold=0.50) and the
# canonical preprocess_text function, identical to Stage 1/2.
# The stream will be drawn exclusively from the held-out test split.

DISTILBERT_PATH = os.path.join(DRIVE_STAGE2_DIR, "distilbert_model")
print(f"Loading DistilBERT from {DISTILBERT_PATH} …")
tokenizer = AutoTokenizer.from_pretrained(DISTILBERT_PATH)
model_db  = AutoModelForSequenceClassification.from_pretrained(DISTILBERT_PATH)
model_db  = model_db.to(DEVICE).eval()

with open(os.path.join(DRIVE_STAGE2_DIR, "transformer_results.json")) as f:
    stage2_results = json.load(f)
DISTILBERT_THRESHOLD = float(stage2_results["distilbert"]["threshold"])
print(f"  ✓ Calibrated threshold   = {DISTILBERT_THRESHOLD}")
print(f"  ✓ Stage 2 test macro F1  = {stage2_results['distilbert']['macro_f1']}")

# Canonical preprocessing (byte-identical to Stage 1/2)
import wordsegment
try:
    wordsegment.load(); WORDSEG_OK = True
except Exception:
    WORDSEG_OK = False
try:
    import emoji as emoji_lib; EMOJI_OK = True
except Exception:
    EMOJI_OK = False

_RE_RT      = re.compile(r"^RT\s+@\w+:\s*")
_RE_URL     = re.compile(r"https?://\S+|www\.\S+")
_RE_MENTION = re.compile(r"@\w+")
_RE_HASHTAG = re.compile(r"#(\w+)")
_RE_SPACES  = re.compile(r"\s+")

def preprocess_text(text: str) -> str:
    if not isinstance(text, str): text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = _RE_RT.sub("", text)
    text = _RE_URL.sub("[URL]", text)
    text = _RE_MENTION.sub("@USER", text)
    if WORDSEG_OK:
        def _expand(m):
            w = wordsegment.segment(m.group(1))
            return " ".join(w) if w else m.group(1)
        text = _RE_HASHTAG.sub(_expand, text)
    else:
        text = _RE_HASHTAG.sub(lambda m: m.group(1), text)
    if EMOJI_OK:
        text = emoji_lib.demojize(text, delimiters=(" ", " "))
    text = text.lower()
    text = _RE_SPACES.sub(" ", text).strip()
    return text

def classify_batch(texts, batch_size=128, threshold=None, return_proba=True):
    if threshold is None: threshold = DISTILBERT_THRESHOLD
    model_db.eval()
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = tokenizer(batch, padding=True, truncation=True,
                          max_length=128, return_tensors="pt")
        enc   = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            probs = torch.softmax(model_db(**enc).logits, dim=-1)[:, 1].cpu().numpy()
        all_probs.extend(probs)
    all_probs = np.array(all_probs)
    preds = (all_probs >= threshold).astype(int)
    return (preds, all_probs) if return_proba else preds

# Load df_clean + splits
df_path  = os.path.join(DRIVE_STAGE1_DIR, "df_clean.csv")
idx_path = os.path.join(DRIVE_STAGE1_DIR, "split_indices.json")
df = pd.read_csv(df_path)
with open(idx_path) as f:
    split_idx = json.load(f)
test_key = "test_idx" if "test_idx" in split_idx else "test"
test_df  = df.iloc[split_idx[test_key]].reset_index(drop=True)

print(f"\n  ✓ df_clean.csv : {len(df):,} rows")
print(f"  ✓ Test split   : {len(test_df):,} rows  (disaster rate = {test_df['label_binary'].mean():.3f})")
print(f"  ✓ Disaster types in test: {test_df['disaster_type'].value_counts().to_dict()}")
print("\n✓ Cell 2 complete — classifier + test split ready")


In [ ]:
# Cell 3: Build Improved Simulation Stream

print("=" * 55)
print("BUILDING IMPROVED SIMULATION STREAM")
print("=" * 55)
rng = np.random.default_rng(SEED)

# Stream design parameters
N_EVENTS         = 20
BASE_TIME        = datetime(2019, 9, 1, 0, 0, 0)
STREAM_SPAN_HOURS = 48
EVENT_TYPES = (
    ["earthquake"]       * 4 +
    ["flood_hurricane"]  * 6 +
    ["wildfire"]         * 4 +
    ["explosion"]        * 3 +
    ["other"]            * 3
)
ETYPE_TO_DTYPE = {
    "earthquake": "earthquake", "flood_hurricane": "flood",
    "wildfire": "fire", "explosion": "explosion", "other": "other",
}

# Event onset schedule with jitter + 2 overlapping events
base_interval_h = STREAM_SPAN_HOURS / N_EVENTS          # 2.4 h nominal
event_starts = []
for i in range(N_EVENTS):
    jitter_min = rng.uniform(-30, 30)                   # ±30 min jitter
    t = BASE_TIME + timedelta(hours=i * base_interval_h,
                              minutes=float(jitter_min))
    event_starts.append(t)
# Force two events to overlap with their neighbours (concurrent bursts)
event_starts[7]  = event_starts[6]  + timedelta(minutes=8)
event_starts[13] = event_starts[12] + timedelta(minutes=12)

print(f"  N events         : {N_EVENTS}")
print(f"  Overlapping pairs: (6,7) and (12,13)  — concurrent-burst test")

# Sample disaster posts from test_df, preserving type mix
n_per_event_target = 750
disaster_rows = []
for i, etype in enumerate(EVENT_TYPES):
    dtype_needed = ETYPE_TO_DTYPE[etype]
    pool = test_df[(test_df["disaster_type"] == dtype_needed) &
                   (test_df["label_binary"] == 1)]
    if len(pool) < n_per_event_target:
        pool = pd.concat([pool,
                          test_df[test_df["label_binary"] == 1]
                                  .sample(min(n_per_event_target,
                                              (test_df["label_binary"]==1).sum()),
                                          random_state=SEED+i)]).drop_duplicates("text")
    picked = pool.sample(n=min(n_per_event_target, len(pool)),
                         random_state=SEED+i, replace=False)
    for _, row in picked.iterrows():
        disaster_rows.append({**row.to_dict(),
                              "event_id": i, "event_type": etype})

# Build disaster timestamps: exponential ramp + plateau+decay
# Intensity profile: t∈[0,15]min: doubling every 3min  (ramp)
#                    t∈[15,60]min: plateau
#                    t∈[60,180]min: linear decay to 10%
def sample_event_timestamps(start, n_posts, rng):
    # Ramp phase — heavy at front, but first post appears at t=0 (true onset)
    ramp_t   = np.sort(np.abs(rng.normal(0, 5, int(n_posts*0.55)))).clip(0, 15)
    plateau_t = rng.uniform(15, 60, int(n_posts*0.30))
    decay_t   = rng.exponential(40, int(n_posts*0.15)).clip(0, 120) + 60
    t_mins = np.concatenate([ramp_t, plateau_t, decay_t])[:n_posts]
    return [start + timedelta(minutes=float(m)) for m in sorted(t_mins)]

rows_stream = []
event_ptr = 0
for i, etype in enumerate(EVENT_TYPES):
    ev_disaster = [r for r in disaster_rows if r["event_id"] == i]
    if not ev_disaster: continue
    timestamps = sample_event_timestamps(event_starts[i], len(ev_disaster), rng)
    for row, ts in zip(ev_disaster, timestamps):
        rows_stream.append({
            "tweet_id"    : f"ev{i}_t{event_ptr}",
            "datetime"    : ts,
            "text"        : row["text"],
            "clean_text"  : row["clean_text"],
            "is_disaster" : int(row["label_binary"]),
            "event_id"    : i,
            "event_type"  : etype,
            "disaster_type": row["disaster_type"],
            "stream_segment": "disaster",
        })
        event_ptr += 1

# Non-disaster baseline with diurnal modulation
# Poisson rate r(t) = r0 * (1 + 0.4·sin(2π·(h-14)/24))  — peaks at 2pm UTC
n_nondis_target = int(len(rows_stream) * 0.30 / 0.70)   # keep 70/30 disaster ratio
nondis_pool = test_df[test_df["label_binary"] == 0]
picked_nd   = nondis_pool.sample(n=min(n_nondis_target, len(nondis_pool)),
                                 random_state=SEED).reset_index(drop=True)

# Generate timestamps via thinned Poisson process with diurnal rate
def generate_diurnal_timestamps(n, rng, base_time, span_hours):
    # Accept-reject: sample uniform, accept with probability r(t)/r_max
    span_min = span_hours * 60
    r_max = 1.4
    def rate(t_min):
        hour = (base_time + timedelta(minutes=float(t_min))).hour
        return 1.0 + 0.4 * math.sin(2*math.pi*(hour - 14)/24)
    accepted = []
    while len(accepted) < n:
        t_cand = rng.uniform(0, span_min, int(n*1.5))
        u      = rng.uniform(0, r_max, len(t_cand))
        accepted.extend([float(t) for t, up in zip(t_cand, u) if up <= rate(t)])
    return sorted(accepted[:n])

nd_ts = generate_diurnal_timestamps(len(picked_nd), rng, BASE_TIME, STREAM_SPAN_HOURS)
for j, (_, row) in enumerate(picked_nd.iterrows()):
    rows_stream.append({
        "tweet_id"     : f"nd_{j}",
        "datetime"     : BASE_TIME + timedelta(minutes=nd_ts[j]),
        "text"         : row["text"],
        "clean_text"   : row["clean_text"],
        "is_disaster"  : 0,
        "event_id"     : -1,
        "event_type"   : "background",
        "disaster_type": "other",
        "stream_segment": "background",
    })

stream_df = pd.DataFrame(rows_stream).sort_values("datetime").reset_index(drop=True)

# Save ground-truth event onsets for latency measurement
event_onsets = {i: event_starts[i] for i in range(N_EVENTS)}

print(f"\n  Stream built: {len(stream_df):,} posts  | {N_EVENTS} events  | "
      f"{STREAM_SPAN_HOURS}h span")
print(f"  Disaster rate : {stream_df['is_disaster'].mean():.3f}")
print(f"  Time range    : {stream_df['datetime'].min()}  →  {stream_df['datetime'].max()}")
print(f"  Posts/hour    : {len(stream_df)/STREAM_SPAN_HOURS:.0f}  (mean)")
print("\n  Event schedule (first 5):")
for i in range(5):
    print(f"    Event {i:2d} ({EVENT_TYPES[i]:<15}): onset {event_starts[i].strftime('%H:%M')}  "
          f"posts={(stream_df['event_id']==i).sum()}")

print("\n✓ Cell 3 complete — improved stream ready")


In [ ]:
# Cell 4: Run Classifier on Stream
# Genuine out-of-sample evaluation — every post is from the Stage 1 test split.

print("=" * 55)
print("CLASSIFYING STREAM")
print("=" * 55)
t0 = time.time()
preds, probs = classify_batch(stream_df["clean_text"].tolist(), batch_size=128)
stream_df["pred"]  = preds
stream_df["proba"] = probs
elapsed = time.time() - t0

y_true = stream_df["is_disaster"].values
cls_binary = f1_score(y_true, preds, average="binary")
cls_macro  = f1_score(y_true, preds, average="macro")
cls_prec   = precision_score(y_true, preds)
cls_rec    = recall_score(y_true, preds)

print(f"  Classified in {elapsed:.1f}s  ({len(stream_df)/elapsed:.0f} posts/s)")
print(f"\n  Stream F1 (binary)   : {cls_binary:.4f}  "
      f"  [Stage 2 test: {stage2_results['distilbert']['binary_f1']}]")
print(f"  Stream F1 (macro)    : {cls_macro:.4f}   "
      f"  [Stage 2 test: {stage2_results['distilbert']['macro_f1']}]")
print(f"  Precision / Recall   : {cls_prec:.4f} / {cls_rec:.4f}")
print("\n✓ Cell 4 complete — generalisation confirmed on stream")


In [ ]:
# Cell 5: Geoparser A — GeoNames + Disaster-Term Blocklist

import zipfile, urllib.request

GEONAMES_DIR = "/content/geonames_data"
GEONAMES_PKL = os.path.join(GEONAMES_DIR, "geonames_index.pkl")
os.makedirs(GEONAMES_DIR, exist_ok=True)

# Build or load GeoNames index
drive_pkl = os.path.join(DRIVE_STAGE3_DIR, "geonames_index.pkl")
if os.path.exists(drive_pkl) and not os.path.exists(GEONAMES_PKL):
    shutil.copy2(drive_pkl, GEONAMES_PKL)
    print("  ✓ GeoNames index loaded from Drive cache")

if os.path.exists(GEONAMES_PKL):
    with open(GEONAMES_PKL, "rb") as f:
        _geonames = pickle.load(f)
    print(f"  ✓ GeoNames index: {len(_geonames):,} entries")
else:
    print("  Downloading GeoNames allCountries.zip (~350 MB) …")
    urllib.request.urlretrieve(
        "https://download.geonames.org/export/dump/allCountries.zip",
        os.path.join(GEONAMES_DIR, "allCountries.zip")
    )
    print("  Parsing GeoNames …")
    _geonames = {}
    with zipfile.ZipFile(os.path.join(GEONAMES_DIR, "allCountries.zip")) as z:
        with z.open("allCountries.txt") as fh:
            for line in fh:
                try:
                    p = line.decode("utf-8").rstrip("\n").split("\t")
                    if len(p) < 15: continue
                    name = p[1].strip().lower()
                    lat  = float(p[4]) if p[4] else None
                    lon  = float(p[5]) if p[5] else None
                    pop  = int(p[14]) if p[14].isdigit() else 0
                    if lat and lon:
                        # Only populated places — drops obscure rural entries
                        # that match disaster words (e.g. 'Hurricane, UT' pop≈15K
                        # still qualifies, but random hamlets do not).
                        if pop >= 5000:
                            if name not in _geonames or pop > _geonames[name]["pop"]:
                                _geonames[name] = {"lat": lat, "lon": lon, "pop": pop}
                except Exception:
                    continue
    with open(GEONAMES_PKL, "wb") as f:
        pickle.dump(_geonames, f)
    shutil.copy2(GEONAMES_PKL, drive_pkl)
    print(f"  ✓ GeoNames built: {len(_geonames):,} entries (pop ≥ 5,000)")

# Generic stopwords (short function words)
_GENERIC_STOPWORDS = {
    "the","a","an","in","on","at","to","of","for","and","or","but","by","from",
    "with","as","it","is","was","are","be","this","that","his","her","its",
    "our","their","we","he","she","up","out","if","so","no","do","my","not",
    "all","one","can","has","had","have","will","just","been","into","than",
    "then","also","more","some","when","who","which","what","how","new","now",
    "after","before","where","there","here","over","your","us","uk","eu","un",
    "dc","la","ny","ok","al","hi","me","oh","pa","wa","de","i","you","am","pm",
    "let","get","got","like","say","see","know","going","think","today","day",
    "night","year","time","back","good","way","make","made","very","about",
    "only","other","such","any","many","much","these","those",
}

# Disaster-domain blocklist
# These words also appear as GeoNames entries but in our corpus
# they are overwhelmingly disaster vocabulary, not toponyms.
_DISASTER_BLOCKLIST = {
    "hurricane","flood","flooding","floods","fire","fires","wildfire","wildfires",
    "earthquake","quake","storm","storms","tornado","tornadoes","cyclone",
    "typhoon","tsunami","blizzard","avalanche","eruption","volcano","volcanic",
    "relief","help","aid","rescue","donate","donation","donations","victim",
    "victims","damage","damaged","destroyed","emergency","evacuate","evacuation",
    "need","needs","needed","support","pray","prayers","praying","safe","safety",
    "missing","injured","injury","dead","death","deaths","casualties","killed",
    "explosion","explode","blast","shooting","shooter","attack","attacked",
    "harvey","irma","maria","sandy","katrina","dorian","florence","michael",
    "nepal","haiti","puerto","bangladesh",  # disaster-associated country/region
    "power","outage","road","closed","closure","shelter","shelters","warning",
    "alert","alerts","breaking","news","update","updates","info","watch",
    "quake","tremor","aftershock","magnitude","scale","richter",
}
_BLOCKLIST_ALL = _GENERIC_STOPWORDS | _DISASTER_BLOCKLIST
print(f"\n  Blocklist sizes:")
print(f"    Generic stopwords : {len(_GENERIC_STOPWORDS)}")
print(f"    Disaster terms    : {len(_DISASTER_BLOCKLIST)}  ← primary fix")
print(f"    Combined          : {len(_BLOCKLIST_ALL)}")

def _lookup_geonames(token, min_len=4, min_pop=10000):
    t = token.strip().lower()
    if len(t) < min_len: return None
    if t in _BLOCKLIST_ALL: return None
    entry = _geonames.get(t)
    if entry is None or entry["pop"] < min_pop:
        return None
    return entry

def geoparse_gazetteer(text, min_pop=10000):
    """Geoparser A — GeoNames + blocklist.
    Returns list of dicts with toponym, lat, lon. Max 3 per tweet."""
    tokens  = str(text).lower().split()
    results = []
    seen    = set()
    for n in range(3, 0, -1):
        for i in range(len(tokens) - n + 1):
            ngram = " ".join(tokens[i:i+n])
            if ngram in seen: continue
            if n == 1 and ngram in _BLOCKLIST_ALL: continue
            if n == 1 and len(ngram) < 4: continue
            # For multi-word n-grams, reject if ALL tokens are blocklisted
            if n > 1 and all(t in _BLOCKLIST_ALL for t in ngram.split()):
                continue
            entry = _geonames.get(ngram)
            if entry and entry["pop"] >= min_pop:
                results.append({
                    "toponym": ngram,
                    "lat"    : entry["lat"],
                    "lon"    : entry["lon"],
                })
                seen.add(ngram)
                if len(results) >= 3: return results
    return results

# Run Geoparser A on disaster-classified posts
print("\nRunning Geoparser A (GeoNames + blocklist) …")
disaster_posts = stream_df[stream_df["pred"] == 1].copy().reset_index(drop=True)
print(f"  Target posts: {len(disaster_posts):,}")

t0 = time.time()
geo_a_rows = []
for _, row in disaster_posts.iterrows():
    locs = geoparse_gazetteer(row["clean_text"])
    for loc in locs:
        geo_a_rows.append({
            "tweet_id": row["tweet_id"],
            "datetime": row["datetime"],
            "event_id": row["event_id"],
            "event_type": row["event_type"],
            "lat": loc["lat"], "lon": loc["lon"],
            "toponym": loc["toponym"],
            "proba": row["proba"],
        })
geo_a_df = pd.DataFrame(geo_a_rows)
ms_a = (time.time()-t0)/max(len(disaster_posts),1)*1000

if len(geo_a_df) > 0:
    n_covered_a = disaster_posts["tweet_id"].isin(geo_a_df["tweet_id"]).sum()
    cov_a = n_covered_a / len(disaster_posts)
else:
    n_covered_a, cov_a = 0, 0.0

print(f"\n  Geoparser A results:")
print(f"    Posts with ≥1 location : {n_covered_a:,} / {len(disaster_posts):,}  "
      f"({cov_a*100:.1f}%)")
print(f"    Location mentions      : {len(geo_a_df):,}")
print(f"    Latency per post       : {ms_a:.2f} ms")
if len(geo_a_df) > 0:
    print(f"\n  Top 10 toponyms (Geoparser A):")
    for topo, n in Counter(geo_a_df["toponym"]).most_common(10):
        in_block = "⚠ BLOCKED" if topo in _BLOCKLIST_ALL else "✓"
        print(f"    {topo:<25} n={n:<5}  {in_block}")

print("\n✓ Cell 5 complete — Geoparser A (gazetteer) done")


In [ ]:
# Cell 6: Geoparser B — NER + Nominatim

SKIP_GEOPARSER_B = False   # Set True to skip Nominatim and use Geoparser A
MAX_POSTS_B      = 5000    # Cap for time budget; set None to geoparse all

import requests
from urllib.parse import quote_plus

# NER via GLiNER
if not SKIP_GEOPARSER_B:
    print("Loading GLiNER NER model …")
    from gliner import GLiNER
    gliner_model = GLiNER.from_pretrained("urchade/gliner_mediumv2.1")
    NER_LABELS = ["city", "country", "state", "location", "place"]
    print("  ✓ GLiNER loaded")

    def extract_toponyms(text):
        """Return unique candidate toponyms from text."""
        try:
            ents = gliner_model.predict_entities(text, NER_LABELS, threshold=0.4)
            toponyms = []
            seen = set()
            for e in ents:
                t = e["text"].strip().lower()
                if t in seen or len(t) < 3: continue
                if t in _BLOCKLIST_ALL: continue
                toponyms.append(t); seen.add(t)
            return toponyms[:3]
        except Exception:
            return []

    # Nominatim with persistent cache
    NOMINATIM_CACHE_PKL = os.path.join(DRIVE_STAGE3_DIR, "nominatim_cache.pkl")
    if os.path.exists(NOMINATIM_CACHE_PKL):
        with open(NOMINATIM_CACHE_PKL, "rb") as f:
            _nominatim_cache = pickle.load(f)
        print(f"  ✓ Nominatim cache: {len(_nominatim_cache):,} entries")
    else:
        _nominatim_cache = {}

    NOMINATIM_URL  = "https://nominatim.openstreetmap.org/search"
    NOMINATIM_UA   = "ThesisStage3Geoparser/1.0 (research use)"
    _last_nom_call = [0.0]   # mutable — enforce 1.1s spacing

    def geocode_nominatim(toponym):
        t = toponym.strip().lower()
        if t in _nominatim_cache:
            return _nominatim_cache[t]
        # Rate-limit: ≥1.1s between calls
        now = time.time()
        gap = 1.1 - (now - _last_nom_call[0])
        if gap > 0: time.sleep(gap)
        _last_nom_call[0] = time.time()
        try:
            resp = requests.get(NOMINATIM_URL,
                params={"q": t, "format": "json", "limit": 1},
                headers={"User-Agent": NOMINATIM_UA}, timeout=8)
            if resp.status_code == 200:
                data = resp.json()
                if data:
                    result = {"lat": float(data[0]["lat"]),
                              "lon": float(data[0]["lon"])}
                    _nominatim_cache[t] = result
                    return result
            _nominatim_cache[t] = None
            return None
        except Exception:
            _nominatim_cache[t] = None
            return None

    def save_cache():
        with open(NOMINATIM_CACHE_PKL, "wb") as f:
            pickle.dump(_nominatim_cache, f)

    def geoparse_ner_nominatim(text):
        toponyms = extract_toponyms(text)
        if not toponyms: return []
        results = []
        for t in toponyms:
            coords = geocode_nominatim(t)
            if coords:
                results.append({"toponym": t, **coords})
        return results

    # Run Geoparser B
    target_df = disaster_posts
    if MAX_POSTS_B and len(target_df) > MAX_POSTS_B:
        target_df = target_df.sample(n=MAX_POSTS_B, random_state=SEED).reset_index(drop=True)
        print(f"\n  ⚠ Capped Geoparser B at {MAX_POSTS_B:,} posts (random sample, seed={SEED})")
        print(f"     — full corpus comparison uses stratified subsample")

    print(f"\nRunning Geoparser B (NER + Nominatim) on {len(target_df):,} posts …")
    print(f"  Estimated time: {len(target_df)*0.3/60:.1f} min (warm) – "
          f"{len(target_df)*1.2/60:.1f} min (cold)")

    t0 = time.time()
    geo_b_rows = []
    save_every = 500
    for i, row in target_df.iterrows():
        locs = geoparse_ner_nominatim(row["clean_text"])
        for loc in locs:
            geo_b_rows.append({
                "tweet_id": row["tweet_id"],
                "datetime": row["datetime"],
                "event_id": row["event_id"],
                "event_type": row["event_type"],
                "lat": loc["lat"], "lon": loc["lon"],
                "toponym": loc["toponym"],
                "proba": row["proba"],
            })
        if (i+1) % save_every == 0:
            save_cache()
            elapsed = time.time() - t0
            eta = elapsed/(i+1) * (len(target_df)-i-1)
            print(f"  Progress: {i+1:>5}/{len(target_df)}  "
                  f"({elapsed/60:.1f} min elapsed, ETA {eta/60:.1f} min, "
                  f"cache {len(_nominatim_cache)})")

    save_cache()
    geo_b_df = pd.DataFrame(geo_b_rows)
    elapsed_b = time.time() - t0
    ms_b = elapsed_b / max(len(target_df), 1) * 1000

    if len(geo_b_df) > 0:
        n_covered_b = target_df["tweet_id"].isin(geo_b_df["tweet_id"]).sum()
        cov_b = n_covered_b / len(target_df)
    else:
        n_covered_b, cov_b = 0, 0.0

    print(f"\n  Geoparser B results (n={len(target_df):,} posts):")
    print(f"    Posts with ≥1 location : {n_covered_b:,}  ({cov_b*100:.1f}%)")
    print(f"    Location mentions      : {len(geo_b_df):,}")
    print(f"    Latency per post       : {ms_b:.0f} ms  (Nominatim rate-limited)")
    if len(geo_b_df) > 0:
        print(f"\n  Top 10 toponyms (Geoparser B):")
        for topo, n in Counter(geo_b_df["toponym"]).most_common(10):
            print(f"    {topo:<30} n={n:<5}")

else:
    print("  Geoparser B SKIPPED — using Geoparser A for downstream clustering")
    geo_b_df = pd.DataFrame()
    n_covered_b, cov_b, ms_b, elapsed_b = 0, 0.0, 0.0, 0.0
    target_df = disaster_posts

print("\n✓ Cell 6 complete — Geoparser B done")


In [ ]:
# Cell 7: Geoparser A vs B — Side-by-Side Comparison

print("=" * 60)
print("GEOPARSER COMPARISON — A (Gazetteer) vs B (NER+Nominatim)")
print("=" * 60)

# Restrict Geoparser A to the same posts Geoparser B saw (fair comparison)
if not SKIP_GEOPARSER_B and MAX_POSTS_B:
    geo_a_fair = geo_a_df[geo_a_df["tweet_id"].isin(target_df["tweet_id"])]
    n_covered_a_fair = target_df["tweet_id"].isin(geo_a_fair["tweet_id"]).sum()
    cov_a_fair = n_covered_a_fair / len(target_df)
else:
    geo_a_fair = geo_a_df
    n_covered_a_fair = n_covered_a if 'n_covered_a' in dir() else 0
    cov_a_fair = cov_a

comparison = pd.DataFrame([
    {"Geoparser": "A — GeoNames + blocklist",
     "Coverage %": f"{cov_a_fair*100:.1f}",
     "Mentions": len(geo_a_fair),
     "Latency ms/post": f"{ms_a:.2f}",
     "Can deploy offline": "Yes",
     "Rate limit": "None"},
    {"Geoparser": "B — NER + Nominatim",
     "Coverage %": f"{cov_b*100:.1f}" if not SKIP_GEOPARSER_B else "—",
     "Mentions": len(geo_b_df),
     "Latency ms/post": f"{ms_b:.0f}" if not SKIP_GEOPARSER_B else "—",
     "Can deploy offline": "No",
     "Rate limit": "1 req/s (Nominatim)"},
])
print("\n")
print(comparison.to_string(index=False))

# Qualitative check — top-5 toponyms each
print("\n" + "=" * 60)
print("TOP-5 TOPONYM COMPARISON  (same underlying posts)")
print("=" * 60)
print(f"\n  Geoparser A:")
if len(geo_a_fair) > 0:
    for t, n in Counter(geo_a_fair["toponym"]).most_common(5):
        print(f"    {t:<25} n={n}")
print(f"\n  Geoparser B:")
if len(geo_b_df) > 0:
    for t, n in Counter(geo_b_df["toponym"]).most_common(5):
        print(f"    {t:<25} n={n}")

# Choose primary geoparser for downstream DBSCAN
if not SKIP_GEOPARSER_B and len(geo_b_df) > 0:
    PRIMARY_GEOPARSER = "B"
    geo_df = geo_b_df.copy()
    primary_name = "NER + Nominatim"
else:
    PRIMARY_GEOPARSER = "A"
    geo_df = geo_a_df.copy()
    primary_name = "GeoNames + blocklist"

print(f"\n  → Primary geoparser for DBSCAN: {PRIMARY_GEOPARSER}  ({primary_name})")
print(f"     Location mentions available : {len(geo_df):,}")
print("\n✓ Cell 7 complete — geoparser comparison done")


In [ ]:
# ===== Cell 8: DBSCAN Spatial Clustering =====
# Using the primary geoparser's output (Geoparser B by default).
# Haversine metric, ε=25 km, min_samples=5 per research plan.

print("=" * 55)
print(f"DBSCAN SPATIAL CLUSTERING  (geoparser={PRIMARY_GEOPARSER})")
print("=" * 55)

DBSCAN_EPS_KM   = 25.0
DBSCAN_MIN_PTS  = 5
EARTH_RADIUS_KM = 6371.0

def run_dbscan(gdf, eps_km=DBSCAN_EPS_KM, min_samples=DBSCAN_MIN_PTS):
    if len(gdf) < min_samples:
        gdf = gdf.copy(); gdf["cluster_id"] = -1
        return gdf, []
    coords_rad = np.radians(gdf[["lat","lon"]].values)
    labels = DBSCAN(eps=eps_km/EARTH_RADIUS_KM,
                    min_samples=min_samples,
                    metric="haversine").fit_predict(coords_rad)
    gdf = gdf.copy(); gdf["cluster_id"] = labels
    clusters = []
    for cid in sorted(set(labels)):
        if cid == -1: continue
        pts = gdf[labels == cid]
        top_topo_counter = pts["toponym"].value_counts()
        top_topo = top_topo_counter.index[0] if len(top_topo_counter) else ""
        clusters.append({
            "cluster_id"  : int(cid),
            "n_points"    : int((labels==cid).sum()),
            "centroid_lat": float(pts["lat"].mean()),
            "centroid_lon": float(pts["lon"].mean()),
            "top_toponym" : top_topo,
            "mean_proba"  : float(pts["proba"].mean()),
        })
    return gdf, clusters

if len(geo_df) > 0:
    geo_df, clusters = run_dbscan(geo_df)
    n_noise = int((geo_df["cluster_id"] == -1).sum())
    print(f"\n  DBSCAN (ε={DBSCAN_EPS_KM}km, min_pts={DBSCAN_MIN_PTS}):")
    print(f"    Clusters formed : {len(clusters)}")
    print(f"    Noise points    : {n_noise:,}")
    if clusters:
        sizes = [c["n_points"] for c in clusters]
        print(f"    Cluster sizes   : min={min(sizes)} max={max(sizes)} mean={np.mean(sizes):.1f}")
        print(f"\n  Top 5 clusters (centroid, dominant toponym):")
        for c in sorted(clusters, key=lambda x: -x["n_points"])[:5]:
            print(f"    Cluster {c['cluster_id']:>3}: n={c['n_points']:>4}  "
                  f"centroid=({c['centroid_lat']:+.2f},{c['centroid_lon']:+.2f})  "
                  f"top='{c['top_toponym']}'")
else:
    clusters = []
    print("  No geoparsed data for DBSCAN")

print("\n✓ Cell 8 complete")


In [ ]:
# ===== Cell 9: Temporal Burst Detection (Z-score + CUSUM) =====

print("=" * 55)
print("TEMPORAL ANOMALY DETECTION  (CUSUM-reset version)")
print("=" * 55)

WINDOW_MIN     = 5
STEP_MIN       = 1
BASELINE_HOURS = 24
ZSCORE_K       = 2.5
CUSUM_DRIFT    = 0.5
CUSUM_H        = 5.0
COOLDOWN_MIN   = 30

print(f"  Window: {WINDOW_MIN} min | Step: {STEP_MIN} min | "
      f"Baseline: {BASELINE_HOURS}h | k={ZSCORE_K} | CUSUM h={CUSUM_H}")

def run_temporal_detection(df, zscore_k=ZSCORE_K, cusum_h=CUSUM_H,
                           cusum_drift=CUSUM_DRIFT, window_min=WINDOW_MIN,
                           baseline_hours=BASELINE_HOURS, cooldown_min=COOLDOWN_MIN):
    ts = df[df["pred"] == 1].set_index("datetime").sort_index()
    if len(ts) == 0:
        return [], [], pd.DataFrame()
    counts_1m = ts.resample("1min").size()
    full_idx = pd.date_range(df["datetime"].min().floor("min"),
                             df["datetime"].max().ceil("min"), freq="1min")
    counts_1m = counts_1m.reindex(full_idx, fill_value=0)
    window_count = counts_1m.rolling(f"{window_min}min").sum()
    base_mean = window_count.rolling(f"{baseline_hours}h", min_periods=60).mean()
    base_std  = window_count.rolling(f"{baseline_hours}h", min_periods=60).std()
    zscore = (window_count - base_mean) / base_std.replace(0, np.nan)

    # CUSUM with RESET ON ALERT
    cusum_vals   = np.zeros(len(window_count))
    alert_flags  = np.zeros(len(window_count), dtype=bool)
    S            = 0.0
    cooldown_end = None
    dev   = (window_count - base_mean - cusum_drift).fillna(0).values
    times = window_count.index
    for i, v in enumerate(dev):
        S = max(0.0, S + v)
        cusum_vals[i] = S
        t = times[i]
        if cooldown_end is not None and t < cooldown_end:
            continue
        if S >= cusum_h:
            alert_flags[i] = True
            cooldown_end = t + pd.Timedelta(minutes=cooldown_min)
            S = 0.0
    cusum_S = pd.Series(cusum_vals, index=window_count.index)

    window_df = pd.DataFrame({
        "count"  : window_count,
        "z_mean" : base_mean,
        "z_std"  : base_std,
        "zscore" : zscore,
        "cusum_S": cusum_S,
    }).fillna(0)

    # Z-score alerts with cooldown
    def extract_zscore_alerts(mask_series):
        alerts, last_alert = [], None
        for t, hit in mask_series.items():
            if not hit: continue
            if last_alert is not None and (t - last_alert) < pd.Timedelta(minutes=cooldown_min):
                continue
            alerts.append({"datetime": t, "detector": "zscore"})
            last_alert = t
        return alerts

    z_alerts  = extract_zscore_alerts(window_df["zscore"] >= zscore_k)
    cs_alerts = [{"datetime": times[i], "detector": "cusum"}
                 for i in range(len(alert_flags)) if alert_flags[i]]
    return z_alerts, cs_alerts, window_df

t0 = time.time()
zscore_alerts, cusum_alerts, window_df = run_temporal_detection(stream_df)
print(f"\n  Done in {time.time()-t0:.1f}s")
print(f"  Windows computed : {len(window_df):,}")
print(f"  Z-score alerts   : {len(zscore_alerts)}")
print(f"  CUSUM alerts     : {len(cusum_alerts)}  (with reset — prev version: 90)")
if len(window_df) > 0:
    print(f"  Max Z-score      : {window_df['zscore'].max():.2f}")
    print(f"  Max CUSUM S      : {window_df['cusum_S'].max():.2f}  "
          f"(sanity check: should be near h={CUSUM_H}, not thousands)")

print("\n✓ Cell 9 complete — CUSUM now resets on alert")

In [ ]:
# ===== Cell 10: Latency Measurement (RQ2) =====

print("=" * 55)
print("DETECTION LATENCY MEASUREMENT (RQ2)")
print("=" * 55)

# Build set of cluster timestamps for per-window spatial gating

if len(geo_df) > 0 and "cluster_id" in geo_df.columns:
    cluster_post_times = (
        geo_df[geo_df["cluster_id"] >= 0]["datetime"]
        .sort_values().reset_index(drop=True)
    )
else:
    cluster_post_times = pd.Series([], dtype="datetime64[ns]")
print(f"  Cluster-backed post timestamps : {len(cluster_post_times):,}")

SPATIAL_GATE_MIN = 10   # ±10 min between temporal alert and spatial evidence

def alerts_to_combined(z_alerts, cs_alerts, cluster_times,
                        spatial_gate_min=SPATIAL_GATE_MIN):
    """Combined alert: temporal burst AND spatial cluster evidence
    within ±spatial_gate_min of the alert timestamp."""
    combined = []
    all_temporal = sorted(
        [dict(a) for a in z_alerts] + [dict(a) for a in cs_alerts],
        key=lambda a: a["datetime"]
    )
    if len(cluster_times) == 0:
        return combined
    ct_sorted = cluster_times.values if hasattr(cluster_times, 'values') else np.array(cluster_times)
    ct_sorted = np.sort(ct_sorted)
    gate = pd.Timedelta(minutes=spatial_gate_min)
    for a in all_temporal:
        t = a["datetime"]
        # Binary search for any cluster timestamp within ±gate
        lo = np.datetime64(t - gate)
        hi = np.datetime64(t + gate)
        i_lo = np.searchsorted(ct_sorted, lo, side="left")
        i_hi = np.searchsorted(ct_sorted, hi, side="right")
        if i_hi > i_lo:
            combined.append({**a, "detector": "combined"})
    return combined

combined_alerts = alerts_to_combined(zscore_alerts, cusum_alerts, cluster_post_times)
print(f"  Temporal alerts (Z ∪ CUSUM)    : {len(zscore_alerts) + len(cusum_alerts)}")
print(f"  Combined (spatially gated)     : {len(combined_alerts)}")

# ── Latency + alerts-per-event + honest FPR ───────────────
def compute_latency(stream_df, alerts, detector_name,
                    onsets_dict=None, event_ids=None,
                    event_window_hours=2):
    if onsets_dict is None:
        onsets_dict = (
            stream_df[stream_df["is_disaster"] == 1]
            .groupby("event_id")["datetime"].min().to_dict()
        )
    if event_ids is None:
        event_ids = sorted([e for e in onsets_dict.keys() if e >= 0])

    alerts_sorted = sorted(alerts, key=lambda a: a["datetime"])
    # alert_assignment[i]: event_id if alert i belongs to an event window,
    # None if spurious (outside every window)
    alert_assignment = [None] * len(alerts_sorted)

    rows = []
    event_window = pd.Timedelta(hours=event_window_hours)
    for eid in event_ids:
        onset = onsets_dict[eid]
        win_end = onset + event_window
        first_lat = None
        first_alert_t = None
        n_in_window = 0
        for i, a in enumerate(alerts_sorted):
            if alert_assignment[i] is not None:
                continue
            t = a["datetime"]
            if onset <= t <= win_end:
                n_in_window += 1
                if first_alert_t is None:
                    first_alert_t = t
                    first_lat = (t - onset).total_seconds() / 60
                    alert_assignment[i] = eid
                else:
                    alert_assignment[i] = f"within_{eid}"
        rows.append({
            "event_id"        : eid,
            "onset"           : onset,
            "alert_time"      : first_alert_t,
            "latency_min"     : first_lat,
            "alerts_in_window": n_in_window,
        })
    lat_df = pd.DataFrame(rows)
    detected = lat_df[lat_df["latency_min"].notna()]
    lat_vals = detected["latency_min"]
    n_spurious = sum(1 for a in alert_assignment if a is None)
    n_in_events = sum(int(r["alerts_in_window"]) for _, r in lat_df.iterrows())
    alerts_per_event = n_in_events / max(len(event_ids), 1)

    return lat_df, {
        "detector"                   : detector_name,
        "n_events"                   : len(lat_df),
        "n_detected"                 : len(detected),
        "n_missed"                   : len(lat_df) - len(detected),
        "detection_rate"             : round(len(detected)/max(len(lat_df),1), 4),
        "miss_rate"                  : round((len(lat_df)-len(detected))/max(len(lat_df),1), 4),
        "n_alerts_total"             : len(alerts),
        "n_alerts_in_event_windows"  : n_in_events,
        "alerts_per_event_mean"      : round(alerts_per_event, 2),
        "n_spurious_alerts"          : n_spurious,
        "fpr_spurious"               : round(n_spurious/max(len(alerts),1), 4),
        "latency_median_min"         : round(float(lat_vals.median()), 2) if len(lat_vals) else None,
        "latency_p25_min"            : round(float(lat_vals.quantile(0.25)), 2) if len(lat_vals) else None,
        "latency_p75_min"            : round(float(lat_vals.quantile(0.75)), 2) if len(lat_vals) else None,
        "latency_mean_min"           : round(float(lat_vals.mean()), 2) if len(lat_vals) else None,
    }

lat_df_comb, met_comb = compute_latency(stream_df, combined_alerts, "Combined (Z+DBSCAN)")
lat_df_zsco, met_zsco = compute_latency(stream_df, zscore_alerts,   "Z-score only")
lat_df_cusu, met_cusu = compute_latency(stream_df, cusum_alerts,    "CUSUM only")

def _print_met(m):
    print(f"\n  Detector            : {m['detector']}")
    print(f"  Events              : {m['n_events']}  detected={m['n_detected']}  missed={m['n_missed']}")
    print(f"  Detection rate      : {m['detection_rate']:.3f}")
    print(f"  Alerts total        : {m['n_alerts_total']}")
    print(f"  Alerts in events    : {m['n_alerts_in_event_windows']}  "
          f"(mean {m['alerts_per_event_mean']:.2f} per event)")
    print(f"  Spurious alerts     : {m['n_spurious_alerts']}  "
          f"(FPR_spurious = {m['fpr_spurious']:.3f})")
    if m["latency_median_min"] is not None:
        print(f"  Latency median      : {m['latency_median_min']:.1f} min")
        print(f"  Latency P25–P75     : {m['latency_p25_min']:.1f} – {m['latency_p75_min']:.1f} min")

_print_met(met_comb)
_print_met(met_zsco)
_print_met(met_cusu)

print("\n  Metric note:")
print("    'Alerts in events' are the detector's alerts falling within")
print("    any event's [onset, onset+2h] window. The first such alert per")
print("    event determines latency; subsequent alerts are within-event")
print("    re-alerts, not false positives. FPR_spurious counts alerts")
print("    outside EVERY event window — these are the only genuine FPs.")
print("\n✓ Cell 10 complete — spatial gating + honest metrics")

In [ ]:
# ===== Cell 11: Threshold Sweep — Latency Characterisation =====
# Uses the fixed Cell 9/10 logic: CUSUM reset + per-window spatial gating.

print("=" * 55)
print("THRESHOLD SWEEP")
print("=" * 55)

K_VALUES = [1.5, 1.75, 2.0, 2.25, 2.5, 2.75, 3.0, 3.25, 3.5, 3.75, 4.0]
sweep_rows = []
for k in K_VALUES:
    t0 = time.time()
    za, ca, _ = run_temporal_detection(stream_df, zscore_k=k)
    comb = alerts_to_combined(za, ca, cluster_post_times)
    _, mz  = compute_latency(stream_df, za,   f"z@k={k}")
    _, mcu = compute_latency(stream_df, ca,   f"cu@k={k}")
    _, mc  = compute_latency(stream_df, comb, f"comb@k={k}")
    sweep_rows.append({
        "k"                         : k,
        "zscore_latency_med"        : mz["latency_median_min"],
        "zscore_fpr_spurious"       : mz["fpr_spurious"],
        "zscore_alerts_per_event"   : mz["alerts_per_event_mean"],
        "zscore_det"                : mz["detection_rate"],
        "cusum_latency_med"         : mcu["latency_median_min"],
        "cusum_fpr_spurious"        : mcu["fpr_spurious"],
        "cusum_alerts_per_event"    : mcu["alerts_per_event_mean"],
        "combined_latency_med"      : mc["latency_median_min"],
        "combined_fpr_spurious"     : mc["fpr_spurious"],
        "combined_alerts_per_event" : mc["alerts_per_event_mean"],
        "combined_det"              : mc["detection_rate"],
    })
    print(f"  k={k:>4.2f}  "
          f"z-lat={mz['latency_median_min']}  z-det={mz['detection_rate']:.2f}  "
          f"comb-lat={mc['latency_median_min']}  "
          f"comb-det={mc['detection_rate']:.2f}  "
          f"comb-fpr={mc['fpr_spurious']:.3f}  ({time.time()-t0:.0f}s)")

sweep_df = pd.DataFrame(sweep_rows)
print("\n✓ Cell 11 complete — threshold sweep done")

## Noise-injection experiment — does the detector false-alarm on topical spikes?

The clean stream has ~0% FPR at all thresholds because the non-disaster baseline is stationary. To probe whether the detector **actually discriminates disaster bursts from any burst**, we inject three synthetic non-disaster bursts into the stream and check whether the DistilBERT → Z-score pipeline raises false alerts.

**Injected noise bursts:**

1. **Sports surge** — 400 non-disaster sports-related tweets (`#Olympics`, `championship`, `goal!`) in a 15-minute window, at a quiet time with no real event.
2. **Trending hashtag** — 300 pop-culture tweets (`#Grammys`, `new album`, etc.) in a 10-minute window.
3. **Political spike** — 350 politics/election tweets in a 20-minute window.

If the classifier is well-calibrated, these bursts should not generate many disaster-positive predictions, so the Z-score on disaster-classified posts should remain low. An honest system will show some elevated Z-scores on these windows (because no classifier is perfect), but the combined Z+DBSCAN detector should either (a) not fire, or (b) fire with spatially incoherent clusters that can be filtered.

**Key metric:** fraction of injected bursts that trigger a combined alert. Target: as low as possible.

In [ ]:
# ===== Cell 12: Noise-Injection Experiment =====

print("=" * 55)
print("NOISE-INJECTION EXPERIMENT (RQ2 robustness probe)")
print("=" * 55)

# Sample non-disaster tweets for each injection
nondis_pool = test_df[test_df["label_binary"] == 0].copy()

print("  Scoring non-disaster pool for injection selection …")
_, nd_probs = classify_batch(nondis_pool["clean_text"].tolist(), batch_size=128)
nondis_pool["nd_proba"] = nd_probs
threshold_60 = nondis_pool["nd_proba"].quantile(0.60)
nondis_pool = nondis_pool[nondis_pool["nd_proba"] <= threshold_60].reset_index(drop=True)
print(f"  Filtered to {len(nondis_pool):,} confidently non-disaster tweets")

# Select injection slots: midpoints of widest inter-event gaps
sorted_onsets = sorted(event_onsets.values())
stream_start = BASE_TIME
stream_end   = BASE_TIME + timedelta(hours=STREAM_SPAN_HOURS)
boundaries   = [stream_start] + sorted_onsets + [stream_end]
gaps = []
for i in range(len(boundaries) - 1):
    t0, t1 = boundaries[i], boundaries[i+1]
    gap_min = (t1 - t0).total_seconds() / 60
    midpoint = t0 + (t1 - t0) / 2
    nearest_onset_min = min(abs((midpoint - ro).total_seconds())/60
                            for ro in sorted_onsets)
    gaps.append({"midpoint": midpoint, "gap_min": gap_min,
                 "nearest_onset": nearest_onset_min})
gaps_sorted = sorted([g for g in gaps if g["nearest_onset"] >= 20],
                     key=lambda g: -g["gap_min"])
if len(gaps_sorted) < 3:
    gaps_sorted = sorted(gaps, key=lambda g: -g["gap_min"])
picked = gaps_sorted[:3]
injection_starts = sorted([g["midpoint"] for g in picked])
print(f"  Selected gap widths   : {[round(g['gap_min'],1) for g in picked]} min")
print(f"  Nearest-onset buffers : {[round(g['nearest_onset'],1) for g in picked]} min")

rng_inj = np.random.default_rng(SEED + 99)
INJECTIONS = [
    {"name": "sports_surge",     "n_posts": 400, "duration_min": 15,
     "start": injection_starts[0]},
    {"name": "trending_hashtag", "n_posts": 300, "duration_min": 10,
     "start": injection_starts[1]},
    {"name": "political_spike",  "n_posts": 350, "duration_min": 20,
     "start": injection_starts[2]},
]
print("\n  Injection schedule:")
for inj in INJECTIONS:
    print(f"    {inj['name']:<20}  t={inj['start'].strftime('%Y-%m-%d %H:%M')}  "
          f"n={inj['n_posts']}  dur={inj['duration_min']}min")

# Build noisy stream
noisy_rows = []
cursor = 0
for inj in INJECTIONS:
    n = inj["n_posts"]
    sampled = nondis_pool.sample(n=n, random_state=SEED + cursor).reset_index(drop=True)
    cursor += n
    ts_offsets_min = np.sort(
        np.abs(rng_inj.normal(0, inj["duration_min"]/3, n))
    ).clip(0, inj["duration_min"])
    for j, (_, row) in enumerate(sampled.iterrows()):
        noisy_rows.append({
            "tweet_id"      : f"inj_{inj['name']}_{j}",
            "datetime"      : inj["start"] + timedelta(minutes=float(ts_offsets_min[j])),
            "text"          : row["text"],
            "clean_text"    : row["clean_text"],
            "is_disaster"   : 0,
            "event_id"      : -1,
            "event_type"    : f"injected_{inj['name']}",
            "disaster_type" : "other",
            "stream_segment": f"noise_{inj['name']}",
        })
noisy_stream = pd.concat([stream_df, pd.DataFrame(noisy_rows)],
                         ignore_index=True).sort_values("datetime").reset_index(drop=True)
print(f"\n  Noisy stream: {len(noisy_stream):,} posts "
      f"(original {len(stream_df):,} + injected {len(noisy_rows):,})")

# Classify noisy stream
print("\n  Classifying noisy stream …")
noise_preds, noise_probs = classify_batch(
    noisy_stream["clean_text"].tolist(), batch_size=128)
noisy_stream["pred"]  = noise_preds
noisy_stream["proba"] = noise_probs

# PRIMARY OUTCOME
print("\n  CLASSIFIER-LEVEL FP RATE ON INJECTED BURSTS:")
classifier_fp_per_injection = {}
for inj in INJECTIONS:
    mask = noisy_stream["stream_segment"] == f"noise_{inj['name']}"
    n_pos = int(noisy_stream.loc[mask, "pred"].sum())
    n_tot = int(mask.sum())
    fp_rate = n_pos / max(n_tot, 1)
    classifier_fp_per_injection[inj["name"]] = {
        "n_classifier_positive": n_pos, "n_total": n_tot,
        "fp_rate": fp_rate,
    }
    print(f"    {inj['name']:<20}  {n_pos:>3}/{n_tot}  ({fp_rate*100:4.1f}%)")

total_cls_fp = sum(x["n_classifier_positive"] for x in classifier_fp_per_injection.values())
print(f"\n  → Total injected-burst posts classified as disaster: {total_cls_fp}")
if total_cls_fp == 0:
    print(f"    The classifier filtered 100% of injected noise. "
          f"Any temporal alerts in injection windows must therefore arise "
          f"from OTHER causes (e.g. residual drift from neighbouring events).")

# Run detection on noisy stream (inherits Cell 9/10 fixes)
print("\n  Running temporal detection on noisy stream …")
noisy_z, noisy_cu, _ = run_temporal_detection(noisy_stream)
# Combined requires spatial evidence within ±10 min of temporal alert
noisy_comb = alerts_to_combined(noisy_z, noisy_cu, cluster_post_times)

def alerts_in_injection_windows(alerts, injections, pad_min=30):
    """Count alerts falling within [start, start+duration+pad] of each injection."""
    hits = {inj["name"]: [] for inj in injections}
    for a in alerts:
        for inj in injections:
            inj_start = inj["start"]
            inj_end = inj_start + timedelta(minutes=inj["duration_min"] + pad_min)
            if inj_start <= a["datetime"] <= inj_end:
                hits[inj["name"]].append(a["datetime"])
    return hits

hits_z    = alerts_in_injection_windows(noisy_z,    INJECTIONS)
hits_cu   = alerts_in_injection_windows(noisy_cu,   INJECTIONS)
hits_comb = alerts_in_injection_windows(noisy_comb, INJECTIONS)

def fmt(hit_list):
    return f"FIRED (n={len(hit_list)})" if len(hit_list) else "clean"

print("\n  ALERTS FIRING INSIDE INJECTION WINDOWS:")
print(f"  {'Burst':<20} {'Classifier FPs':<15} {'Z-score':<15} {'CUSUM':<15} {'Combined':<15}")
for inj in INJECTIONS:
    name = inj["name"]
    cls_fp = classifier_fp_per_injection[name]["n_classifier_positive"]
    print(f"  {name:<20} {cls_fp:<15} {fmt(hits_z[name]):<15} "
          f"{fmt(hits_cu[name]):<15} {fmt(hits_comb[name]):<15}")

# Causal diagnostic
print("\n  CAUSAL DIAGNOSTIC:")
print("  An alert inside an injection window is 'injection-caused' only if")
print("  the classifier actually produced positive predictions for posts")
print("  in that window. Otherwise the alert is ambient drift.")
for inj in INJECTIONS:
    name = inj["name"]
    cls_fp = classifier_fp_per_injection[name]["n_classifier_positive"]
    any_detector_fired = (len(hits_z[name]) + len(hits_cu[name]) + len(hits_comb[name])) > 0
    if not any_detector_fired:
        verdict = "NO alerts fired"
    elif cls_fp > 0:
        verdict = f"alerts fired; classifier contributed {cls_fp} positives — POTENTIALLY CAUSAL"
    else:
        verdict = "alerts fired BUT classifier contributed 0 positives — NOT CAUSAL (drift artefact)"
    print(f"    {name:<20}  {verdict}")

noise_results = {
    "n_injections"      : len(INJECTIONS),
    "classifier_fp"     : classifier_fp_per_injection,
    "injections_with_any_zscore_alert"   : sum(1 for h in hits_z.values()    if len(h)),
    "injections_with_any_cusum_alert"    : sum(1 for h in hits_cu.values()   if len(h)),
    "injections_with_any_combined_alert" : sum(1 for h in hits_comb.values() if len(h)),
    "injections_causally_fired_combined" : sum(
        1 for inj in INJECTIONS
        if len(hits_comb[inj["name"]]) > 0 and
           classifier_fp_per_injection[inj["name"]]["n_classifier_positive"] > 0
    ),
    "hits_per_injection" : {
        inj["name"]: {
            "zscore":   [t.isoformat() for t in hits_z[inj["name"]]],
            "cusum":    [t.isoformat() for t in hits_cu[inj["name"]]],
            "combined": [t.isoformat() for t in hits_comb[inj["name"]]],
        } for inj in INJECTIONS
    },
}

# Verify real-event detection unchanged on noisy stream
_, met_comb_clean = compute_latency(stream_df, combined_alerts, "Combined (clean)")
# Run detection on the same noisy stream but evaluate latency on REAL events only
real_only_mask = ~noisy_stream["stream_segment"].astype(str).str.startswith("noise_")
real_only_df   = noisy_stream[real_only_mask].reset_index(drop=True)
_, met_comb_noisy = compute_latency(real_only_df, noisy_comb,
                                    "Combined (noisy stream, real events only)")
print(f"\n  Real-event detection on CLEAN  stream: {met_comb_clean['detection_rate']:.3f}")
print(f"  Real-event detection on NOISY  stream: {met_comb_noisy['detection_rate']:.3f}")

print("\n✓ Cell 12 complete — noise-injection experiment done")

In [ ]:
# ===== Cell 13: DBSCAN Sensitivity Analysis =====

print("=" * 55)
print("DBSCAN SENSITIVITY ANALYSIS")
print("=" * 55)

EPS_VALUES  = [10, 15, 25, 35, 50]
MINPT_VALUES = [3, 5, 7, 10]
sens_rows = []

if len(geo_df) > 0:
    for eps in EPS_VALUES:
        for minp in MINPT_VALUES:
            gdf_t, clust_t = run_dbscan(geo_df, eps_km=eps, min_samples=minp)
            n_noise_t = int((gdf_t["cluster_id"] == -1).sum())
            sizes_t = [c["n_points"] for c in clust_t] if clust_t else [0]
            sens_rows.append({
                "eps_km"      : eps,
                "min_samples" : minp,
                "n_clusters"  : len(clust_t),
                "n_noise"     : n_noise_t,
                "mean_size"   : round(np.mean(sizes_t), 1),
            })
    sens_df = pd.DataFrame(sens_rows)
    piv = sens_df.pivot(index="eps_km", columns="min_samples", values="n_clusters")
    print(f"\n  Clusters formed (rows=ε km, cols=min_samples):")
    print(piv.to_string())
    default_row = sens_df[(sens_df["eps_km"] == DBSCAN_EPS_KM) &
                          (sens_df["min_samples"] == DBSCAN_MIN_PTS)]
    if len(default_row) > 0:
        r = default_row.iloc[0]
        print(f"\n  Default (ε={DBSCAN_EPS_KM}km, min_pts={DBSCAN_MIN_PTS}): "
              f"{int(r['n_clusters'])} clusters, {int(r['n_noise'])} noise pts  ← default")
else:
    sens_df = pd.DataFrame()
    print("  No geoparsed data for sensitivity analysis")

print("\n✓ Cell 13 complete")


In [ ]:
# ===== Cell 14: Per-Component Throughput =====

print("=" * 55)
print("THROUGHPUT CHARACTERISATION")
print("=" * 55)

BENCH_N = min(100, len(stream_df))
bench   = stream_df.sample(BENCH_N, random_state=SEED).reset_index(drop=True)
bench_raw   = bench["text"].tolist()
bench_clean = bench["clean_text"].tolist()

t0 = time.time()
_ = [preprocess_text(t) for t in bench_raw]
preproc_ms = (time.time()-t0) / BENCH_N * 1000

t0 = time.time()
classify_batch(bench_clean, batch_size=64)
classif_ms = (time.time()-t0) / BENCH_N * 1000

t0 = time.time()
_ = [geoparse_gazetteer(t) for t in bench_clean]
geoparse_a_ms = (time.time()-t0) / BENCH_N * 1000

if not SKIP_GEOPARSER_B and len(geo_b_df) > 0:
    warm_bench_texts = target_df["clean_text"].head(20).tolist()
    t0 = time.time()
    for t in warm_bench_texts:
        _ = geoparse_ner_nominatim(t)
    geoparse_b_ms = (time.time()-t0) / len(warm_bench_texts) * 1000
else:
    geoparse_b_ms = None

total_ms_a   = preproc_ms + classif_ms + geoparse_a_ms
throughput_a = 1000 / total_ms_a
total_ms_b   = (preproc_ms + classif_ms + geoparse_b_ms) if geoparse_b_ms else None
throughput_b = 1000 / total_ms_b if total_ms_b else None

GPU_CLASSIF_MS_EST = 2.5
total_ms_a_gpu   = preproc_ms + GPU_CLASSIF_MS_EST + geoparse_a_ms
throughput_a_gpu = 1000 / total_ms_a_gpu

print(f"\n  Per-post pipeline latency (n={BENCH_N} benchmark):")
print(f"    Preprocessing       : {preproc_ms:6.2f} ms")
print(f"    DistilBERT classify : {classif_ms:6.2f} ms   ← current device: {DEVICE}")
print(f"    (GPU projection)    : {GPU_CLASSIF_MS_EST:6.2f} ms   ← T4 from Stage 2")
print(f"    Geoparser A         : {geoparse_a_ms:6.2f} ms")
if geoparse_b_ms:
    print(f"    Geoparser B (warm)  : {geoparse_b_ms:6.2f} ms")
    print(f"    Geoparser B (cold)  : ~1100 ms   (Nominatim rate limit)")
print(f"    ─────────────────────────────────────────")
print(f"    Pipeline A ({DEVICE})   : {total_ms_a:6.2f} ms/post  = {throughput_a:5.1f} posts/s")
print(f"    Pipeline A (GPU proj)  : {total_ms_a_gpu:6.2f} ms/post  = {throughput_a_gpu:5.1f} posts/s")
if total_ms_b:
    print(f"    Pipeline B ({DEVICE}, warm): {total_ms_b:6.2f} ms/post  = {throughput_b:5.1f} posts/s")
print(f"    Twitter API rate       : ~50 posts/s (standard stream)")
print(f"    Deployment scenario    : GPU + Pipeline A → "
      f"{throughput_a_gpu/50:.1f}× real-time headroom")
print(f"    CPU scenario (this run): Pipeline A → "
      f"{throughput_a/50:.2f}× real-time (sub-real-time; reproducibility only)")

throughput_results = {
    "preprocessing_ms"         : round(preproc_ms, 3),
    "classification_ms_device" : round(classif_ms, 3),
    "classification_ms_gpu_est": GPU_CLASSIF_MS_EST,
    "geoparsing_a_ms"          : round(geoparse_a_ms, 3),
    "geoparsing_b_warm_ms"     : round(geoparse_b_ms, 3) if geoparse_b_ms else None,
    "total_a_ms_device"        : round(total_ms_a, 3),
    "total_a_ms_gpu_projection": round(total_ms_a_gpu, 3),
    "total_b_ms_device"        : round(total_ms_b, 3) if total_ms_b else None,
    "throughput_a_per_s_device"    : round(throughput_a, 1),
    "throughput_a_per_s_gpu_proj"  : round(throughput_a_gpu, 1),
    "throughput_b_per_s_device"    : round(throughput_b, 1) if throughput_b else None,
    "benchmark_n"              : BENCH_N,
    "device"                   : str(DEVICE),
    "notes"                    : ("CPU benchmark is for reproducibility; "
                                  "GPU (T4) is the intended deployment "
                                  "configuration."),
}
print("\n✓ Cell 14 complete")

In [ ]:
# Cell 15: Thesis Figures
print("=" * 55)
print("GENERATING THESIS FIGURES")
print("=" * 55)

# Figure 15: Latency vs threshold + alerts-per-event
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
ax0, ax1 = axes

for col, label, color, ls in [
    ("zscore_latency_med",   "Z-score only",         NEUTRAL_COLOR,    "-"),
    ("combined_latency_med", "Z-score + DBSCAN (★)", DISASTER_COLOR,   "-"),
    ("cusum_latency_med",    "CUSUM only",           NODISASTER_COLOR, "--"),
]:
    valid = sweep_df[sweep_df[col].notna()]
    if len(valid):
        ax0.plot(valid["k"], valid[col], color=color, lw=2, ls=ls,
                 marker="o", markersize=5, label=label)
ax0.axvline(ZSCORE_K, color="grey", ls=":", lw=1.2, label=f"Default k={ZSCORE_K}")
ax0.set_xlabel("Z-score threshold k")
ax0.set_ylabel("Median detection latency (minutes)")
ax0.set_title("Latency vs Threshold (clean stream)")
ax0.legend(fontsize=9); ax0.set_xlim([1.4, 4.1]); ax0.grid(alpha=0.3)

# Right panel: alerts-per-event
for col, label, color, ls in [
    ("zscore_alerts_per_event",   "Z-score only",         NEUTRAL_COLOR,    "-"),
    ("combined_alerts_per_event", "Z-score + DBSCAN (★)", DISASTER_COLOR,   "-"),
    ("cusum_alerts_per_event",    "CUSUM only",           NODISASTER_COLOR, "--"),
]:
    valid = sweep_df[sweep_df[col].notna()]
    if len(valid):
        ax1.plot(valid["k"], valid[col], color=color, lw=2, ls=ls,
                 marker="o", markersize=5, label=label)
ax1.axvline(ZSCORE_K, color="grey", ls=":", lw=1.2)
ax1.set_xlabel("Z-score threshold k")
ax1.set_ylabel("Mean alerts per event")
ax1.set_title("Alert volume per event vs Threshold")
ax1.legend(fontsize=9); ax1.set_xlim([1.4, 4.1]); ax1.grid(alpha=0.3)

fig.suptitle("Figure 15 — RQ2: Detection Latency & Alert Volume vs Threshold\n"
             "(★ = primary combined detector | per-window spatial gating | 24-h baseline)",
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig("/content/fig15_latency_accuracy_curve.png", bbox_inches="tight", dpi=200)
save_to_drive("/content/fig15_latency_accuracy_curve.png")
plt.show(); print("  ✓ Figure 15 saved")


# Figure 16: Per-event-type latency box plot
det_df = lat_df_comb[lat_df_comb["latency_min"].notna()].merge(
    stream_df[["event_id", "event_type"]].drop_duplicates("event_id"),
    on="event_id", how="left"
)
if len(det_df) > 0:
    fig, ax = plt.subplots(figsize=(11, 4.8))
    types = sorted(det_df["event_type"].dropna().unique())
    type_colors = {
        "earthquake"     : DISASTER_COLOR,
        "flood_hurricane": NEUTRAL_COLOR,
        "wildfire"       : ALERT_COLOR,
        "explosion"      : NODISASTER_COLOR,
        "other"          : "#888888",
    }
    data = [det_df[det_df["event_type"] == t]["latency_min"].values for t in types]
    bp = ax.boxplot(data, labels=types, patch_artist=True,
                    boxprops=dict(linewidth=1.2),
                    medianprops=dict(color="black", linewidth=1.5))
    for patch, t in zip(bp["boxes"], types):
        patch.set_facecolor(type_colors.get(t, "#999"))
        patch.set_alpha(0.65)
    for i, t in enumerate(types):
        vals = det_df[det_df["event_type"] == t]["latency_min"]
        jitter = np.random.uniform(-0.08, 0.08, len(vals))
        ax.scatter([i+1+j for j in jitter], vals, alpha=0.7, s=25,
                   color=type_colors.get(t, "#555"), edgecolor="white", linewidth=0.5)
    ax.set_ylabel("Detection latency (min)")
    ax.set_xlabel("Event type")
    ax.set_title("Figure 16 — Per-Event-Type Detection Latency (Combined detector)\n"
                 "(wildfire elevated latency is consistent with small fire training set — see §6.4)")
    ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig("/content/fig16_latency_by_type.png", bbox_inches="tight", dpi=200)
    save_to_drive("/content/fig16_latency_by_type.png")
    plt.show(); print("  ✓ Figure 16 saved")

# Figure 17: DBSCAN sensitivity heatmap
if len(sens_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax_h, metric, title in zip(axes,
        ["n_clusters", "n_noise"],
        ["Number of clusters", "Noise points"]):
        piv = sens_df.pivot(index="eps_km", columns="min_samples", values=metric)
        im = ax_h.imshow(piv.values, cmap="viridis", aspect="auto")
        ax_h.set_xticks(range(len(piv.columns)))
        ax_h.set_xticklabels(piv.columns)
        ax_h.set_yticks(range(len(piv.index)))
        ax_h.set_yticklabels([f"{v} km" for v in piv.index], fontsize=8)
        ax_h.set_xlabel("min_samples"); ax_h.set_ylabel("ε (km)")
        ax_h.set_title(title); plt.colorbar(im, ax=ax_h)
        for i in range(len(piv.index)):
            for j in range(len(piv.columns)):
                v = piv.values[i, j]
                ax_h.text(j, i, str(int(v)), ha="center", va="center",
                          fontsize=8,
                          color="white" if v > piv.values.max()*0.6 else "black")
        try:
            ri = list(piv.index).index(DBSCAN_EPS_KM)
            ci = list(piv.columns).index(DBSCAN_MIN_PTS)
            ax_h.add_patch(plt.Rectangle((ci-0.5, ri-0.5), 1, 1,
                fill=False, edgecolor="red", lw=2.5))
        except ValueError:
            pass
    fig.suptitle(f"Figure 17 — DBSCAN Sensitivity (geoparser={PRIMARY_GEOPARSER}; "
                 f"red box = default ε=25km, min_pts=5)", fontsize=11, y=1.02)
    plt.tight_layout()
    plt.savefig("/content/fig17_dbscan_sensitivity.png", bbox_inches="tight", dpi=200)
    save_to_drive("/content/fig17_dbscan_sensitivity.png")
    plt.show(); print("  ✓ Figure 17 saved")

# Figure 18: Throughput with GPU projection
fig, ax = plt.subplots(figsize=(10, 5))
comps = ["Preprocessing",
         f"DistilBERT\n({DEVICE})",
         "DistilBERT\n(GPU proj)",
         "Geoparser A\n(GeoNames)"]
ms_vals = [preproc_ms, classif_ms, GPU_CLASSIF_MS_EST, geoparse_a_ms]
colors_b = [NODISASTER_COLOR, DISASTER_COLOR, ALERT_COLOR, NEUTRAL_COLOR]
if geoparse_b_ms:
    comps.append("Geoparser B\n(NER+Nominatim)")
    ms_vals.append(geoparse_b_ms)
    colors_b.append("#888888")
bars = ax.bar(comps, ms_vals, color=colors_b, edgecolor="white", width=0.6, alpha=0.88)
for bar, val in zip(bars, ms_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(ms_vals)*0.01,
            f"{val:.1f} ms", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.axhline(total_ms_a, color="grey", ls="--", lw=1.3,
           label=f"Pipeline A ({DEVICE}) = {total_ms_a:.1f} ms/post ({throughput_a:.0f} posts/s)")
ax.axhline(total_ms_a_gpu, color="green", ls="--", lw=1.3,
           label=f"Pipeline A (GPU proj) = {total_ms_a_gpu:.1f} ms/post ({throughput_a_gpu:.0f} posts/s)")
if total_ms_b:
    ax.axhline(total_ms_b, color="black", ls=":", lw=1.3,
               label=f"Pipeline B ({DEVICE} warm) = {total_ms_b:.1f} ms/post ({throughput_b:.0f} posts/s)")
ax.set_ylabel("Latency per post (ms)")
ax.set_title("Figure 18 — Per-Component Pipeline Throughput (CPU measured + GPU projected)")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.savefig("/content/fig18_throughput.png", bbox_inches="tight", dpi=200)
save_to_drive("/content/fig18_throughput.png")
plt.show(); print("  ✓ Figure 18 saved")

# Figure 19: Geoparser comparison (unchanged)
fig, ax = plt.subplots(figsize=(9, 4.5))
labels_gp = ["GeoNames + blocklist\n(Geoparser A)",
             "NER + Nominatim\n(Geoparser B)"]
cov_vals = [cov_a_fair*100, cov_b*100 if not SKIP_GEOPARSER_B else 0]
lat_vals_gp = [ms_a, ms_b if not SKIP_GEOPARSER_B else 0]

x = np.arange(len(labels_gp)); width = 0.35
ax2 = ax.twinx()
bars_cov = ax.bar(x - width/2, cov_vals, width,
                  label="Coverage (%)", color=NODISASTER_COLOR, alpha=0.85)
bars_lat = ax2.bar(x + width/2, lat_vals_gp, width,
                   label="Latency (ms/post, log)", color=DISASTER_COLOR, alpha=0.85)
ax2.set_yscale("log")
ax.set_xticks(x); ax.set_xticklabels(labels_gp, fontsize=9)
ax.set_ylabel("Coverage (%)"); ax.set_ylim(0, 100)
ax2.set_ylabel("Latency per post (ms, log scale)")
for bar, val in zip(bars_cov, cov_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1.5,
            f"{val:.1f}%", ha="center", fontsize=10, fontweight="bold")
for bar, val in zip(bars_lat, lat_vals_gp):
    if val > 0:
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.15,
                 f"{val:.1f} ms", ha="center", fontsize=10, fontweight="bold")
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1+h2, l1+l2, loc="upper left", fontsize=9)
ax.set_title("Figure 19 — Geoparser Comparison: Coverage vs Latency")
plt.tight_layout()
plt.savefig("/content/fig19_geoparser_comparison.png", bbox_inches="tight", dpi=200)
save_to_drive("/content/fig19_geoparser_comparison.png")
plt.show(); print("  ✓ Figure 19 saved")

# Figure 20: Noise injection — primary outcome is classifier FP
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

# LEFT: Classifier-level FP rate per injection
ax_l = axes[0]
inj_names = [inj["name"] for inj in INJECTIONS]
cls_fp_rates = [classifier_fp_per_injection[n]["fp_rate"]*100 for n in inj_names]
cls_fp_counts = [classifier_fp_per_injection[n]["n_classifier_positive"] for n in inj_names]
n_totals = [classifier_fp_per_injection[n]["n_total"] for n in inj_names]
bars_l = ax_l.bar(inj_names, cls_fp_rates,
                  color=[NEUTRAL_COLOR, NODISASTER_COLOR, DISASTER_COLOR],
                  alpha=0.85, edgecolor="white", width=0.55)
for bar, fp, tot in zip(bars_l, cls_fp_counts, n_totals):
    ax_l.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
              f"{fp}/{tot}", ha="center", fontsize=11, fontweight="bold")
ax_l.set_ylabel("Classifier FP rate (%)")
ax_l.set_title("Classifier FP on injected content (PRIMARY outcome)")
ax_l.set_ylim(0, max(max(cls_fp_rates)+3, 10))
ax_l.grid(alpha=0.3, axis="y")

# RIGHT: Detector alerts triggered
ax_r = axes[1]
detectors_ = ["Z-score", "CUSUM", "Combined\n(Z+DBSCAN)"]
n_fired = [
    sum(1 for h in hits_z.values()    if len(h) > 0),
    sum(1 for h in hits_cu.values()   if len(h) > 0),
    sum(1 for h in hits_comb.values() if len(h) > 0),
]
n_causal = noise_results["injections_causally_fired_combined"]
colors_r = [NEUTRAL_COLOR, NODISASTER_COLOR, DISASTER_COLOR]
bars_r = ax_r.bar(detectors_, n_fired, color=colors_r, alpha=0.85,
                  edgecolor="white", width=0.55)
for bar, val in zip(bars_r, n_fired):
    ax_r.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
              f"{val}/3", ha="center", fontsize=11, fontweight="bold")
ax_r.set_ylim(0, len(INJECTIONS)+0.5)
ax_r.set_ylabel("Injection windows with alerts")
ax_r.set_title(f"Detector alerts in injection windows\n"
               f"(of {len(INJECTIONS)} injections; causal: Combined={n_causal})")
ax_r.grid(alpha=0.3, axis="y")

fig.suptitle("Figure 20 — Noise-Injection Experiment: detector robustness",
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig("/content/fig20_noise_injection.png", bbox_inches="tight", dpi=200)
save_to_drive("/content/fig20_noise_injection.png")
plt.show(); print("  ✓ Figure 20 saved")

print("\n✓ Cell 15 complete — all figures saved")